# Прогноз формы сердца по одной фазе КТ

**Задача.** Проверить, можно ли по одной трёхмерной КТ предсказать форму сердца в других, действительно снятых фазах того же исследования Ника. «Одна КТ» здесь означает полный 3D-том фазы 00, а не отдельный срез. Сравнение с фактическими фазами 4D-КТ выполнено для готовой сердечной модели MONAI Physio без обучения на данных проекта.

Постановка исследования описана в [протоколе](21.14_Протокол_прогноза_сердца_по_одной_КТ.ipynb). Ни автоматические проверочные маски, ни номинальные проценты R–R сами по себе не удостоверяют истинное механическое движение отдельных участков миокарда.

## Источники данных и роль каждой фазы

Для Ника имеются 12 фаз 4D-КТ серии 307 и автоматические маски всего сердца TotalSegmentator v2. Фаза 00 с DICOM-меткой 14 % R–R заранее выбрана как индивидуальный вход: по её КТ и маске подогнана готовая модель формы. Фазы 01–08 с номинальными метками 23–92 % служат удержанным контролем прогноза в диапазоне, допустимом для сети. Фазы 09–11 в это сетевое сравнение не входят, поскольку их номинальные значения не меньше 100 %. КТ и маски удержанных фаз использованы только для оценки после фиксации подгонки и вывода сети.

Автоматические маски получены независимо от предсказания сети. Принятой ручной проверки их анатомических границ нет. Неопределённость их границ, различия дыхательного состояния и соответствие номинального времени R–R механической фазе отдельно не оценены. Индивидуальные КТ и поверхности хранятся во внешнем локальном хранилище, а в этот ноутбук включены только обезличенные агрегаты.

## Неподвижная маска как отдельный исходный контроль

Ранее был рассчитан простой контроль для трёх добровольцев: автоматическая маска фазы 00 каждого человека без изменений сопоставлялась со всеми его фазами 01 и далее. Для каждого сравнения получены Dice бинарных объёмов, расстояние между центрами масок в миллиметрах и абсолютная разница объёмов в миллилитрах. Поскольку у неподвижного контроля предсказанное изменение объёма равно нулю, последняя величина равна отличию объёма целевой маски от объёма опорной. Скрипт проверяет согласованность сеток и аффинных матриц, бинарность и непустоту масок; его контрольные суммы позволяют установить конкретные входы.

| Доброволец | Целевых фаз | Средний Dice | Средняя дистанция центров, мм | Минимальный Dice | Наибольшая абсолютная разница объёмов, мл |
|---|---:|---:|---:|---:|---:|
| Adam | 11 | 0,9299 | 1,090 | 0,8786 | 50,919 |
| Georg | 14 | 0,9250 | 1,453 | 0,8918 | 44,565 |
| Nix | 11 | 0,9177 | 1,894 | 0,8521 | 54,404 |

У Ника минимальный Dice пришёлся на фазу 03. Эти числа характеризуют расхождение *автоматических объёмных масок* с неизменной маской фазы 00. Они не являются ошибкой готовой сети. В парном сетевом сравнении ниже исходной служит **другая** неподвижная форма — подогнанный исторический шаблон; сравниваются восемь фаз и другая метрика. Поэтому значение Dice 0,9177 нельзя напрямую сопоставлять с дистанциями в миллиметрах.

## Сетевой прогноз и парное сравнение на восьми фазах

Исторические сердечные веса MONAI Physio/PhysicsNeMo из выпуска 2026.07.3 и полная PCA-модель проверены технически и использованы без дообучения. Индивидуальную форму подогнали только по фазе 00 Ника. В сеть подали 15 коэффициентов формы и номинальную долю R–R целевой фазы; из рассчитанного смещения вычли смещение для исходной доли 0,14. Так получено изменение относительно фазы 00. Отдельные КТ/маски целевых фаз не использовались при подгонке или сетевом выводе.

Парный контроль — **та же подогнанная поверхность**, оставленная неподвижной. Для прогноза и контроля рассчитана симметричная средняя дистанция ближайших **вершин** до вершин автоматической поверхности целевой фазы. Функция `distances()` ищет соседей с помощью KDTree; ближайшую точку внутри треугольника она не ищет. Значение зависит от плотности и распределения вершин и не является точной дистанцией «точка — треугольная поверхность» (ASSD). Итоговые значения вычислены по всем вершинам сохранённых поверхностей, без прореживания; меньшая дистанция означает лучшее совпадение по этой конкретной метрике.

| Фаза | Номинальная доля R–R | Неподвижный шаблон, мм | Прогноз, мм | Прогноз минус контроль, мм |
|---|---:|---:|---:|---:|
| 01 | 0,23 | 4,091 | 4,308 | +0,216 |
| 02 | 0,33 | 4,521 | 4,459 | −0,063 |
| 03 | 0,43 | 4,499 | 4,574 | +0,075 |
| 04 | 0,53 | 4,314 | 4,531 | +0,217 |
| 05 | 0,62 | 4,352 | 4,475 | +0,123 |
| 06 | 0,72 | 4,540 | 4,347 | −0,193 |
| 07 | 0,82 | 4,633 | 4,410 | −0,222 |
| 08 | 0,92 | 5,122 | 4,758 | −0,364 |

Сеть улучшила выбранную метрику на четырёх фазах и ухудшила на четырёх. Среднее по восьми фазам изменилось с **4,509 до 4,483 мм**, то есть на 0,026 мм. Исходная подгонка шаблона к автоматической маске фазы 00 сама имела дистанцию ближайших вершин **4,384 мм**. Пофазные смещения и происхождение модели приведены в [протоколе вычисления](mp06_nix_one_ct_forecast_report.md).

## Интерпретация результата и границы применимости

Этот опыт действительно проверяет готовый прогноз на фазах *той же* 4D-КТ, которые сеть не получала при вычислении формы. Он показывает отсутствие устойчивого выигрыша над неподвижным подогнанным шаблоном по выбранной метрике: четыре улучшения сочетаются с четырьмя ухудшениями, а средняя разница мала относительно исходной ошибки подгонки.

Остаются два разных источника неопределённости. Во-первых, исторический шаблон Duke Heart и маска `whole_heart` могут описывать разные анатомические границы. Во-вторых, отдельные TSv2-маски являются автоматическим сравнительным материалом; их ошибки способны влиять на пофазную дистанцию. Вычисление ближайших вершин не прослеживает траектории анатомически соответствующих точек, а номинальный процент R–R не является независимой ЭКГ-меткой точного механического состояния. Поэтому наблюдаемое небольшое изменение метрики не доказывает точность индивидуального поля движения.

## Решение для методики и воспроизведение

Готовый прогноз на Нике вычислен без обучения на наших данных, однако по проведённой проверке его нельзя считать заменой реально измеренной 4D-КТ или допущенной геометрией для FEM. Перед такой передачей нужны согласованная анатомическая область шаблона и контрольных масок, проверка точности фазовых контуров, качества деформации и сетки, а затем одинаковая электрическая постановка с анализом `Z` и `ΔZ`.

[Скрипт объёмного неподвижного контроля](mp05_07_eval.py) запускается с командой `python mp05_07_eval.py baseline --data-root <локальный_корень_производных_4D_КТ>`. [Скрипт сетевого прогноза](mp06_nix_one_ct_forecast.py) отдельно выполняет подгонку по фазе 00, сетевой вывод и итоговую оценку сохранённых поверхностей; параметры и результаты приведены в [отчёте](mp06_nix_one_ct_forecast_report.md). Первичные медицинские данные и индивидуальные сетки остаются вне Git.